# Replication of Wang's paper

## Preprocessing

Instead of using raw waveforms, the backbone takes mel-spectrograms as input. A mel-spectrogram is a visual representation of the spectrum of frequencies in a sound as they vary with time, adjusted to match how the human ear perceives pitch.

Step 1: Extract the attributes from each file (from attributes_00.csv)

Step 2: Convert the .wav files to mel-spectrograms.

Step 3: The mel-spectrogram is partitioned into $16\times16$ patches.

## Pass to the EAT model

Step 4: Pass the patches through the EAT Encoder, each patch outputs its own embedding containing deep representational info.

Step 5: Apply average pooling to all patch embeddings extracted by the model from one mel-spectrogram (corresponding to 1 audiofile). This results in a single vector (the embedding) that represents the core characteristics of that specific recording.

## Training phase

Classification task: 
The embeddings are fed into a Linear Classifier. The model is trained to classify the machine's attributes (e.g., its specific state or configuration) and/or its domain (source vs. target).

ArcFace loss:
Instead of a standard loss function, the system uses ArcFace loss. This adds a "margin" in the angular space, forcing the model to create highly compact and distinct clusters for normal machine sounds.

Optimization:
This process is done using either Full Fine-Tuning (updating all 88 million parameters) or LoRA (updating only small "low-rank" matrices) to adapt the pre-trained model to the specific DCASE dataset.









## EAT applied to specific machine
Here all the train files from the bearing machine from 2023 are processed. All the 1000 embedding are extracted and saved, together with a .csv file which contains the meta data of each audio file. In this file the file_name, feature_path, domain, and attribute_id are stored. The domain would be source or target and the attribute ID will be a number. For example for the bearing machine it has a velocity and location attribute. If the velocity is 1 and the location is A, it will receive the attribute ID 1. if the velocity is one and the location is B, it will receive the attribute ID 2, etc.

In [1]:
import os
import glob
import pandas as pd
import torch
import torchaudio
import soundfile as sf
import numpy as np
from tqdm import tqdm
from transformers import AutoModel

In [2]:
# --- CONFIGURATION ---
base_data_path = "data"
model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
target_length = 1024
norm_mean = -4.268
norm_std = 4.569

# Initialize Model once outside the loops
print(f"Loading model: {model_id}")
model = AutoModel.from_pretrained(model_id, trust_remote_code=True).eval().cuda()

# 1. Find all DCASE year directories (e.g., dcase2023t2, dcase2024t2)
year_dirs = [d for d in os.listdir(base_data_path) if d.startswith("dcase") and d.endswith("t2")]

for year_folder in year_dirs:
    year_label = year_folder.replace("dcase", "").replace("t2", "") # e.g., "2025"
    
    # We focus on dev_data/raw as per your structure
    dev_raw_path = os.path.join(base_data_path, year_folder, "dev_data", "raw")
    
    if not os.path.exists(dev_raw_path):
        print(f"Skipping {year_folder}: Path {dev_raw_path} not found.")
        continue

    # 2. Find all machine directories in this year
    machines = [m for m in os.listdir(dev_raw_path) if os.path.isdir(os.path.join(dev_raw_path, m))]

    for machine in machines:
        print(f"\n--- Processing Year: {year_label} | Machine: {machine} ---")
        
        machine_path = os.path.join(dev_raw_path, machine)
        train_dir = os.path.join(machine_path, "train")
        attr_csv_path = os.path.join(machine_path, "attributes_00.csv")
        
        output_dir = f"features/Prototype/dcase{year_label}t2/{machine}/features"
        os.makedirs(output_dir, exist_ok=True)

        if not os.path.exists(attr_csv_path) or not os.path.exists(train_dir):
            print(f"Missing files for {machine} in {year_label}, skipping...")
            continue

        attr_df = pd.read_csv(attr_csv_path)
        attr_df['file_name_clean'] = attr_df['file_name'].apply(lambda x: os.path.basename(x))
        feature_cols = [c for c in attr_df.columns if c not in ['file_name', 'file_name_clean']]
        attr_df['unique_attr'] = attr_df[feature_cols].astype(str).apply(lambda x: '_'.join(x), axis=1)
        unique_combinations = attr_df['unique_attr'].unique()
        attr_map = {val: i for i, val in enumerate(unique_combinations)}
        attr_df['attribute_id'] = attr_df['unique_attr'].map(attr_map)
        attr_lookup = attr_df.set_index('file_name_clean')['attribute_id'].to_dict()

        # 3. Processing Loop for the specific machine
        files = [f for f in os.listdir(train_dir) if f.endswith(".wav")]
        metadata = []

        for file_name in tqdm(files, desc=f"{year_label}_{machine}"):
            try:
                # 1. Metadata extraction
                # Using .get() is safer and won't crash if the file is missing from the CSV
                attr_id = attr_lookup.get(file_name)
                
                if attr_id is None:
                    print(f"Warning: {file_name} not found in attributes CSV! Skipping...")
                    continue # This now correctly only triggers if attr_id is missing
                
                # 2. Audio Loading
                path = os.path.join(train_dir, file_name)
                wav, sr = sf.read(path)
                
                # Convert to tensor and move to GPU
                waveform = torch.from_numpy(wav).float().cuda()
                
                if sr != 16000:
                    waveform = torchaudio.functional.resample(waveform, sr, 16000)

                # 3. Feature Extraction Preprocessing
                waveform = waveform - waveform.mean()
                mel = torchaudio.compliance.kaldi.fbank(
                    waveform.unsqueeze(0),
                    htk_compat=True,
                    sample_frequency=16000,
                    window_type='hanning',
                    num_mel_bins=128,
                    frame_shift=10
                )

                # Pad/Truncate to target_length
                if mel.shape[0] < target_length:
                    mel = torch.nn.functional.pad(mel, (0, 0, 0, target_length - mel.shape[0]))
                else:
                    mel = mel[:target_length, :]

                # 4. Model Inference
                mel = (mel - norm_mean) / (norm_std * 2)
                # Input shape needs to be [Batch, Channel, Time, Frequency]
                mel = mel.unsqueeze(0).unsqueeze(0).cuda() 
                
                with torch.no_grad():
                    feat = model.extract_features(mel)
                    # Global average pooling over the time dimension
                    feat = torch.mean(feat, dim=1).squeeze(0).cpu().numpy()

                # 4. Save Mel-spectrograms directly
                mel_name = file_name.replace(".wav", "_mel.pt")
                torch.save(mel.cpu(), os.path.join(output_dir, mel_name))

                metadata.append({
                    "file_name": file_name,
                    "feature_path": mel_name,
                    "domain": "source" if "source" in file_name else "target",
                    "attribute_id": int(attr_id) 
                })

                # # 5. Save and log
                # feat_name = file_name.replace(".wav", ".npy")
                # np.save(os.path.join(output_dir, feat_name), feat)

                # metadata.append({
                #     "file_name": file_name,
                #     "feature_path": feat_name,
                #     "domain": "source" if "source" in file_name else "target",
                #     "attribute_id": int(attr_id) # Ensure clean integer format
                # })

            except Exception as e:
                print(f"Error processing {file_name}: {e}")
                continue

        # Save metadata for this machine
        pd.DataFrame(metadata).to_csv(f"features/Prototype/dcase{year_label}t2/{machine}/train_metadata.csv", index=False)

print("\nAll tasks completed!")

Loading model: worstchan/EAT-base_epoch30_finetune_AS2M

--- Processing Year: 2023 | Machine: bearing ---


2023_bearing:   0%|          | 0/1000 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class ArcFaceClassifier(nn.Module):
    def __init__(self, emb_size, num_classes, s=30.0, m=0.5):
        super().__init__()
        self.s = s # Scaling factor [cite: 55]
        self.m = m # Margin [cite: 55]
        # Weight matrix W where each column is a class embedding [cite: 56]
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, emb_size))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels):
        # Normalize features and weights to get Cosine Theta [cite: 57, 59]
        logits = F.linear(F.normalize(x), F.normalize(self.weight))
        
        # Apply the ArcFace margin [cite: 54]
        theta = torch.acos(torch.clamp(logits, -1.0 + 1e-7, 1.0 - 1e-7))
        target_logits = torch.cos(theta + self.m)
        
        # Replace original logits with margin-adjusted logits for the correct class
        one_hot = torch.zeros_like(logits)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        output = (one_hot * target_logits) + ((1.0 - one_hot) * logits)
        return output * self.s

In [4]:
# Initialization
num_classes = len(unique_combinations) # From your previous preprocessing step
classifier_head = ArcFaceClassifier(emb_size=768, num_classes=num_classes).cuda()

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(classifier_head.parameters()), 
    lr=5e-5
)

criterion = nn.CrossEntropyLoss()

In [13]:
from torch.utils.data import Dataset, DataLoader

class ASDMultiMachineDataset(Dataset):
    def __init__(self, metadata_df, feature_dir):
        self.df = metadata_df
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load the pre-computed Mel-spectrogram
        mel_path = os.path.join(self.feature_dir, row['feature_path'])
        mel = torch.load(mel_path)
        
        if mel.dim() > 3:
            mel = mel.squeeze(0)
            
            attr_label = int(row['attribute_id'])
            domain_label = 0 if row['domain'] == 'source' else 1
        
        return mel.float(), torch.tensor(attr_label).long(), torch.tensor(domain_label).long()

In [14]:
import torch.nn as nn
from peft import LoraConfig, get_peft_model

class WangSystem1Model(nn.Module):
    def __init__(self, base_model, num_attributes, num_domains=2):
        super().__init__()
        
        # 1. Setup LoRA
        lora_config = LoraConfig(
            r=8, 
            lora_alpha=16, 
            target_modules=["qkv", "proj"], # Target attention 
            lora_dropout=0.05,
            bias="none"
        )
        self.backbone = get_peft_model(base_model, lora_config)
        
        # 2. Setup Multi-Task ArcFace Heads 
        self.arcface_attr = ArcFaceClassifier(emb_size=768, num_classes=num_attributes, s=30.0, m=0.5)
        self.arcface_domain = ArcFaceClassifier(emb_size=768, num_classes=num_domains, s=30.0, m=0.5)

    def forward(self, mel, attr_labels=None, domain_labels=None):
        # Pass Mel through LoRA-adapted EAT
        feat = self.backbone.extract_features(mel)
        
        # Global average pooling over the time dimension 
        feat = torch.mean(feat, dim=1) 
        
        # ArcFace margin-adjusted logits [cite: 54]
        logits_attr = self.arcface_attr(feat, attr_labels)
        logits_domain = self.arcface_domain(feat, domain_labels)
        
        return logits_attr, logits_domain

In [15]:
# Assuming you are looping over your 'machines' list:
for machine in machines:
    print(f"--- Fine-Tuning LoRA for {machine} ---")
    
    # 1. Prepare Data
    metadata_path = f"features/Prototype/dcase{year_label}t2/{machine}/train_metadata.csv"
    feature_dir = f"features/Prototype/dcase{year_label}t2/{machine}/features"
    
    df = pd.read_csv(metadata_path)
    num_classes = df['attribute_id'].nunique()
    
    dataset = ASDMultiMachineDataset(df, feature_dir)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    # 2. Initialize Model for this machine
    # Note: Reload base_model from huggingface here to ensure a fresh backbone for each machine
    base_model = AutoModel.from_pretrained(model_id, trust_remote_code=True).cuda()
    model = WangSystem1Model(base_model, num_attributes=num_classes).cuda()
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    
    # 3. Training Loop
    total_steps = 20000 
    current_step = 0
    
    while current_step < total_steps:
        for mels, attr_labels, domain_labels in dataloader:
            if current_step >= total_steps: break
                
            mels = mels.cuda()
            attr_labels = attr_labels.cuda()
            domain_labels = domain_labels.cuda()
            
            optimizer.zero_grad()
            
            # Forward pass happens entirely inside the model now!
            logits_attr, logits_domain = model(mels, attr_labels, domain_labels)
            
            # Multi-task loss 
            loss = criterion(logits_attr, attr_labels) + criterion(logits_domain, domain_labels)
            
            loss.backward()
            optimizer.step()
            
            current_step += 1
            if current_step % 1000 == 0:
                print(f"Machine: {machine} | Step [{current_step}/{total_steps}], Loss: {loss.item():.4f}")
                
    # 4. Save the LoRA weights for this specific machine
    model.backbone.save_pretrained(f"models/eat_lora_{machine}_{year_label}")

--- Fine-Tuning LoRA for bearing ---


C:\Users\20202299\AppData\Local\Temp\ipykernel_26064\4015717022.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mel = torch.load(mel_path)


OutOfMemoryError: CUDA out of memory. Tried to allocate 386.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 10.16 GiB is allocated by PyTorch, and 392.77 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)